In [ ]:
import pathlib
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import torch
import trimesh
from metrics import chamfer_distance

import utils, dataset, visualization, encoder, decoder, latent_decoder

%load_ext autoreload
%autoreload 2

device = torch.device('cuda:0')

In [ ]:
#Load a model:

VAE_experiment_name = "Jun27_10-53_epochs=10000_latent_dim=128_hidden_dim=128_embedding_dim=120"
VAEconfig, VAEmodel_config, VAEencoder, VAEdecoder = utils.reload_model(None, None, VAE_experiment_name, 'best_train', device)

LATENT_experiment_name = ""
LATENTconfig, LATENTmodel_config, LATENTdecoder = utils.reload_model_latent(None, None, LATENT_experiment_name, 'checkpoint', device)

split = 0
match split:
    case 0:
        set_GRASP_CODE = dataset.Dataset_Latent_grasp_and_code('train')
        set_GRASP_PC = dataset.Dataset_Latent_grasp_and_pc('train')
    case 1:
        set_GRASP_CODE = dataset.Dataset_Latent_grasp_and_code('val')
        set_GRASP_PC = dataset.Dataset_Latent_grasp_and_pc('val')
    case 2:
        set_GRASP_CODE = dataset.Dataset_Latent_grasp_and_code('test')
        set_GRASP_PC = dataset.Dataset_Latent_grasp_and_pc('test')

print(len(set_GRASP_CODE))
print(len(set_GRASP_PC))

In [ ]:
number_of_points = 2048
VAE_DDIM_steps = 50
LATENT_DDIM_steps = 50
number_of_DDIM_iterations = 1
visualize = 1

with torch.no_grad():
    
    avg_chamfer = 0.
    for object_index in range(min(len(set_GRASP_CODE), 50)):
        for i in range(number_of_DDIM_iterations):
            
            grasp = set_GRASP_CODE[object_index]['grasp'].to(device).unsqueeze(0)
            mean = set_GRASP_CODE[object_index]['code'].to(device).unsqueeze(0)
            pc = set_GRASP_PC[object_index]['point_cloud'].to(device).unsqueeze(0)
            
            code = latent_decoder.sample_ddim(
                decoder, grasp, LATENTmodel_config['latent_dim'], 
                LATENT_DDIM_steps, LATENTconfig['timesteps'])

            mean_pc = decoder.sample_ddim(
                VAEdecoder, mean, n_points=number_of_points, 
                steps=VAE_DDIM_steps, timesteps=VAEconfig['timesteps']
            )

            code_pc = decoder.sample_ddim(
                VAEdecoder, code, n_points=number_of_points, 
                steps=VAE_DDIM_steps, timesteps=VAEconfig['timesteps']
            )
            
            # Handle Chamfer output dynamically and extract the float (.item())
            dist = chamfer_distance(pc, code_pc).item()
            avg_chamfer1 += dist
            print(f"pc code_pc, Object {object_index}: {dist:.4f}")
            
            dist = chamfer_distance(mean_pc, code_pc).item()
            avg_chamfer2 += dist
            print(f"mean_pc code_pc, Object {object_index}: {dist:.4f}")

            
            if visualize:
                visualization.visualize_comparison(pc, code_pc, window_name="DDIM Target PC (Red) vs Generated (Blue)")
                visualization.visualize_comparison(mean_pc, code_pc, window_name="DDIM Target MEAN_PC (Red) vs Generated (Blue)")
            
            
    print(f"Average Chamfer distance = {avg_chamfer1 / (len(set_GRASP_CODE) * number_of_DDIM_iterations):.5f}")